In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

In [2]:
words = open('../data/names.txt','r').read().splitlines()
string = sorted(list(set((''.join(words)))))



stoi = { s: i + 1 for (i,s) in enumerate(string)}
itos = { i+1 : s for (i,s) in enumerate(string)}
stoi["."] = 0 
itos[0] = '.'

In [35]:
block_size = 8

X, Y = [], []
random.seed(32)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))


def create_dataset(words) : 
    X, Y = [], []

    for w in words : 
        context = [0]*block_size

        for ch in w + '.' : 
            ix = stoi[ch]

            X.append(context)
            Y.append(ix)

            context = context[1:] + [ix]

    return (torch.tensor(X), torch.tensor(Y))

X_train, Y_train = create_dataset(words[:n1])
X_dev, Y_dev = create_dataset(words[n1:n2])
X_test, Y_test = create_dataset(words[n2:])

In [54]:
class Linear : 
    def __init__(self, nin, nout, bias = True):
        self.weight = torch.randn((nin, nout)) / nin**0.5 # for the vairance (kaiming initialization)
        self.bias = torch.randn(nout) if bias else None

    def __call__(self, x):
        self.out = x@self.weight 
        if self.bias is not None: 
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([self.bias] if self.bias is not None else [])



class BatchNorm1D : 
    def __init__(self, dim, eps = 1e-5, momentum = 0.1):
        self.eps = eps
        self.momentum = momentum
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        #trained during training
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

        self.training = True


    def __call__(self, x):
        if self.training : 
            dim = 0 if x.ndim == 2 else (0,1)
            xmean = x.mean(dim, keepdim = True)
            xvar = x.var(dim, keepdim = True)
        else : 
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x - xmean)/(xvar + self.eps)**0.5
        self.out = self.gamma * xhat + self.beta


        # update the buffers 

        if self.training : 
            with torch.no_grad() : 
                self.running_mean = (1 - self.momentum) * self.running_mean +self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out


    def parameters(self) : 
        return [self.gamma, self.beta]


class Tanh : 
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self) : 
        return []


class Embedding : 
    def __init__(self, nin,nout):
        self.weight = torch.randn(nin,nout)

    def __call__(self, x):
        self.out = self.weight[x]
        return self.out

    def parameters(self) : 
        return[self.weight]

class FlatenConsecutive : 
    def __init__(self, n ):
        self.n = n 

    def __call__(self, x):
        B,T,C = x.shape
        x = x.view(B, T//self.n, C*self.n)
        if x.shape[1] == 1 : 
            x = x.squeeze(1)

        self.out = x
        return self.out

    def parameters(self) : 
        return []


In [ ]:
n_emb = 24
n_train = 128


layers = [
    Embedding(27, n_emb), 
    FlatenConsecutive(2), Linear(2*n_emb, n_train, bias = False), BatchNorm1D(n_train), Tanh(),
    FlatenConsecutive(2), Linear(2*n_train, n_train, bias = False), BatchNorm1D(n_train), Tanh(),
    FlatenConsecutive(2), Linear(2*n_train, n_train, bias = False), BatchNorm1D(n_train), Tanh(),
    Linear(n_train, 27)
          ]

# params init  :

with torch.no_grad() : 
    layers[-1].weight *= 0.1

parameters =[p for l in layers for p in l.parameters()]
for p in parameters :
    p.requires_grad = True

In [ ]:
epochs = 200000

size_batch = 32

for i in range(epochs) : 
    ix = torch.randint(0, X_train.shape[0], (size_batch,))
    x = X_train[ix]
    
    for layer in layers : 
        x = layer(x)
        
    loss = F.cross_entropy(x, Y_train[ix])

    for p in parameters : 
        p.grad = None

    loss.backward()

    lr = 0.1 if i < epochs*0.6 else 0.01

    for p in parameters : 
        p.data += -lr*p.grad

    if i % 10000 == 0:
        print(i, loss)  


print(loss.item())
    

1.7193537950515747


In [68]:
@torch.no_grad()
def split_loss(X, Y):
    for layer in layers: layer.training = False
    x = X
    for layer in layers: x = layer(x)
    loss = F.cross_entropy(x, Y)
    for layer in layers: layer.training = True
    return loss.item()

print('train', split_loss(X_train, Y_train))
print('dev  ', split_loss(X_dev, Y_dev))


train 1.9089157581329346
dev   2.0137791633605957
